### TRANSFORMATION WORKFLOW

1. TRIM SPACES
2. RENAME COLUMNS
3. FIX PRICE / SALES_AMOUNT INCONSISTENCIES
4. CLEAN AND CONVERT DATE COLUMNS
5. VALIDATION CHECKS
6. WRITE INTO SILVER

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

In [0]:
# 0 LOAD DATA & READ FROM BRONZE
dfsales = spark.table("acdproj.bronze.crm_sales_details")

In [0]:
# 1) TRIM SPACES
for field in dfsales.schema.fields:
    if isinstance(field.dataType, StringType):
        dfsales = dfsales.withColumn(field.name, F.trim(F.col(field.name)))

In [0]:
# 2) RENAME COLUMNS
RENAME_MAP_SALES = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_key",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}
for old_name, new_name in RENAME_MAP_SALES.items():
    dfsales = dfsales.withColumnRenamed(old_name, new_name)

In [0]:
# 3) FIX PRICE / SALES_AMOUNT INCONSISTENCIES
dfsales = dfsales.withColumn(
    "price",
    F.when(
        (F.col("price").isNull()) | (F.col("price") <= 0),
        F.abs(F.col("sales_amount")) / F.col("quantity")
    ).otherwise(F.col("price"))
)

dfsales = dfsales.withColumn(
    "sales_amount",
    F.when(
        (F.col("sales_amount").isNull()) | (F.col("sales_amount") != F.col("quantity") * F.col("price")),
        F.col("quantity") * F.col("price")
    ).otherwise(F.col("sales_amount"))
)

In [0]:
 # 4) CLEAN AND CONVERT DATE COLUMNS
DATE_COLUMNS = ["order_date", "ship_date", "due_date"]

for date_col in DATE_COLUMNS:
    dfsales = dfsales.withColumn(
        date_col,
        F.when(F.col(date_col) == 0, None).otherwise(F.col(date_col))
    )

for date_col in DATE_COLUMNS:
    dfsales = dfsales.withColumn(
        date_col,
        F.try_to_date(F.col(date_col).cast("string"), F.lit("yyyyMMdd"))
    )

In [0]:
# 5) VALIDATION CHECKS (run before saving)
inconsistent_count = dfsales.filter(
    (F.col("sales_amount") != F.col("quantity") * F.col("price")) |
    F.col("sales_amount").isNull() |
    F.col("price").isNull() |
    (F.col("price") <= 0)
).count()
print(f"Remaining price/sales_amount inconsistencies: {inconsistent_count}")

zero_date_count = dfsales.filter(
    F.col("order_date").isNull() |
    F.col("ship_date").isNull() |
    F.col("due_date").isNull()
).count()
print(f"Rows with a null date (originally 0 or malformed in source): {zero_date_count}")

In [0]:
# 6) SAVE TO SILVER
dfsales.write.mode("overwrite").saveAsTable("acdproj.silver.crm_sales")